# Data Visualisations

Figures for the thesis. Each section is self-contained: set the configuration
in section 2, then run the sections you need.

**Section 4** visualises the ERA5 download region derived from a single day of
trajectory data, with the ERA5 0.25° grid drawn over an OpenStreetMap basemap.

## 1. Imports

In [8]:
import math
from pathlib import Path

import numpy as np
import polars as pl
import plotly.graph_objects as go

## 2. Configuration

`TARGET_DATE` selects which day's trajectory partition is loaded. Set it to
`None` to use every day present in the data.

In [9]:
PROJECT_ROOT = Path("/home/tolgaakcay/projects/AircraftTrajectoryPrediction_Thesis")
DATA_SOURCE = PROJECT_ROOT / "src" / "data" / "raw"
PLOT_DIR = PROJECT_ROOT / "plots"

AIRPORT = "LFBO"
TARGET_DATE = "2024-10-02"

BBOX_MARGIN_DEG = 2.0
GRID_STEP_DEG = 0.25

CLAMP_LAT = (20.0, 75.0)
CLAMP_LON = (-110.0, 45.0)

MAP_STYLE = "open-street-map"
MAP_POINT_SAMPLE = 20000

DETAIL_HALF_WIDTH_DEG = 2.0
OVERVIEW_GRID_EVERY = 20

AIRPORT_LATLON = {
    "LFBO": (43.6293, 1.3638),
    "EDDM": (48.3538, 11.7861),
}

## 3. Load trajectory points for the selected date

Reads the hive-partitioned parquet with an explicit glob so non-parquet files in
the tree are ignored. Coordinates outside the clamp window are separated out
rather than dropped, so their effect on the bounding box can be shown.

In [10]:
def load_points(data_source, airport, target_date=None):
    lf = pl.scan_parquet(
        source=str(Path(data_source) / "**" / "*.parquet"),
        hive_schema={
            "destination": pl.String, "year": pl.Int64,
            "month": pl.Int64, "day": pl.Int64,
        },
        hive_partitioning=True,
        cast_options=pl.ScanCastOptions(datetime_cast="nanosecond-downcast"),
    ).filter(pl.col("destination") == airport)

    if target_date is not None:
        y, m, d = (int(v) for v in target_date.split("-"))
        lf = lf.filter(
            (pl.col("year") == y) & (pl.col("month") == m) & (pl.col("day") == d)
        )

    pts = lf.select(
        pl.col("latitude").explode().alias("lat"),
        pl.col("longitude").explode().alias("lon"),
    ).collect()

    lats = pts.get_column("lat").to_numpy()
    lons = pts.get_column("lon").to_numpy()
    ok = ~(np.isnan(lats) | np.isnan(lons))
    return lats[ok], lons[ok]


all_lats, all_lons = load_points(DATA_SOURCE, AIRPORT, TARGET_DATE)

inside = (
    (all_lats >= CLAMP_LAT[0]) & (all_lats <= CLAMP_LAT[1])
    & (all_lons >= CLAMP_LON[0]) & (all_lons <= CLAMP_LON[1])
)
lats, lons = all_lats[inside], all_lons[inside]
out_lats, out_lons = all_lats[~inside], all_lons[~inside]

print(f"date            : {TARGET_DATE or 'all'}")
print(f"points loaded   : {len(all_lats):,}")
print(f"within clamp    : {len(lats):,}")
print(f"outside clamp   : {len(out_lats):,}")
if len(lats):
    print(f"lat range       : {lats.min():.2f} to {lats.max():.2f}")
    print(f"lon range       : {lons.min():.2f} to {lons.max():.2f}")

date            : 2024-10-02
points loaded   : 71,813
within clamp    : 71,813
outside clamp   : 0
lat range       : 35.78 to 56.03
lon range       : -9.32 to 24.80


## 4. Bounding box and ERA5 grid geometry

The request area is the trajectory extent, padded by `BBOX_MARGIN_DEG` and
snapped outward to the ERA5 grid. Snapping is outward on all four edges so no
trajectory point can fall outside the downloaded region.

In [11]:
def compute_area(lats, lons, margin_deg, step):
    north = math.ceil((lats.max() + margin_deg) / step) * step
    south = math.floor((lats.min() - margin_deg) / step) * step
    west = math.floor((lons.min() - margin_deg) / step) * step
    east = math.ceil((lons.max() + margin_deg) / step) * step
    return [round(north, 2), round(west, 2), round(south, 2), round(east, 2)]


def box_ring(north, west, south, east):
    return ([west, east, east, west, west],
            [north, north, south, south, north])


def grid_lines(north, west, south, east, step, every=1):
    lon_pts, lat_pts = [], []
    for x in np.arange(west, east + step / 2, step * every):
        lon_pts += [float(x), float(x), None]
        lat_pts += [south, north, None]
    for y in np.arange(south, north + step / 2, step * every):
        lon_pts += [west, east, None]
        lat_pts += [float(y), float(y), None]
    return lon_pts, lat_pts


def grid_corners(north, west, south, east, step):
    xs = np.arange(west, east + step / 2, step)
    ys = np.arange(south, north + step / 2, step)
    gx, gy = np.meshgrid(xs, ys)
    return gx.ravel(), gy.ravel()


AREA = compute_area(lats, lons, BBOX_MARGIN_DEG, GRID_STEP_DEG)
NORTH, WEST, SOUTH, EAST = AREA

RAW_N, RAW_S = lats.max(), lats.min()
RAW_W, RAW_E = lons.min(), lons.max()
MAR_N, MAR_S = RAW_N + BBOX_MARGIN_DEG, RAW_S - BBOX_MARGIN_DEG
MAR_W, MAR_E = RAW_W - BBOX_MARGIN_DEG, RAW_E + BBOX_MARGIN_DEG

N_LAT = round((NORTH - SOUTH) / GRID_STEP_DEG)
N_LON = round((EAST - WEST) / GRID_STEP_DEG)

print(f"raw extent  N {RAW_N:7.2f}  W {RAW_W:8.2f}  S {RAW_S:7.2f}  E {RAW_E:7.2f}")
print(f"+ margin    N {MAR_N:7.2f}  W {MAR_W:8.2f}  S {MAR_S:7.2f}  E {MAR_E:7.2f}")
print(f"snapped     N {NORTH:7.2f}  W {WEST:8.2f}  S {SOUTH:7.2f}  E {EAST:7.2f}")
print(f"ERA5 grid   {N_LAT} x {N_LON} = {N_LAT * N_LON:,} cells")

raw extent  N   56.03  W    -9.32  S   35.78  E   24.80
+ margin    N   58.03  W   -11.32  S   33.78  E   26.80
snapped     N   58.25  W   -11.50  S   33.75  E   27.00
ERA5 grid   98 x 154 = 15,092 cells


## 5. Overview map: trajectories and the request area

The full request region over OpenStreetMap. Grid lines are drawn every
`OVERVIEW_GRID_EVERY` steps &mdash; at the true 0.25° spacing they would be solid
at this zoom. Section 6 shows the real grid.

In [16]:
if len(lats) > MAP_POINT_SAMPLE:
    sel = np.linspace(0, len(lats) - 1, MAP_POINT_SAMPLE).astype(int)
    plot_lats, plot_lons = lats[sel], lons[sel]
else:
    plot_lats, plot_lons = lats, lons

fig = go.Figure()

gl_lon, gl_lat = grid_lines(NORTH, WEST, SOUTH, EAST,
                            GRID_STEP_DEG, OVERVIEW_GRID_EVERY)
fig.add_trace(go.Scattermap(
    lat=gl_lat, lon=gl_lon, mode="lines",
    line=dict(width=0.5, color="rgba(80,80,80,0.45)"),
    name=f"ERA5 Grid (Scale: {OVERVIEW_GRID_EVERY} x {GRID_STEP_DEG}\u00b0)",
    hoverinfo="skip"))

fig.add_trace(go.Scattermap(
    lat=plot_lats, lon=plot_lons, mode="markers",
    marker=dict(size=3, color="#2c7fb8", opacity=0.35),
    name=f"Trajectory Points", hoverinfo="skip"))

if len(out_lats):
    fig.add_trace(go.Scattermap(
        lat=out_lats, lon=out_lons, mode="markers",
        marker=dict(size=9, color="red"),
        name=f"Outside Clamp (n={len(out_lats):,})"))

for ring, colour, width, label in [
    (box_ring(RAW_N, RAW_W, RAW_S, RAW_E), "#f28e2b", 2,
     "Trajectory Extent (min/max)"),
    (box_ring(MAR_N, MAR_W, MAR_S, MAR_E), "#59a14f", 2,
     f"+ {BBOX_MARGIN_DEG}\u00b0 Bounding Box Margin"),
    (box_ring(NORTH, WEST, SOUTH, EAST), "#e15759", 3.5,
     f"ERA5 Request"),
]:
    rlon, rlat = ring
    fig.add_trace(go.Scattermap(
        lat=rlat, lon=rlon, mode="lines",
        line=dict(width=width, color=colour), name=label))

if AIRPORT in AIRPORT_LATLON:
    ap_lat, ap_lon = AIRPORT_LATLON[AIRPORT]
    fig.add_trace(go.Scattermap(
        lat=[ap_lat], lon=[ap_lon], mode="markers+text",
        marker=dict(size=14, color="black"),
        text=[AIRPORT], textposition="top right",
        textfont=dict(size=13), name=AIRPORT))

fig.update_layout(
    map=dict(style=MAP_STYLE,
             center=dict(lat=(NORTH + SOUTH) / 2, lon=(EAST + WEST) / 2),
             zoom=2.2),
    margin=dict(r=0, t=52, l=0, b=0), height=680,
    title=dict(text=(f"ERA5 download region &mdash; {AIRPORT}, {TARGET_DATE}"
                     f" &mdash; [N {NORTH}, W {WEST}, S {SOUTH}, E {EAST}],"
                     f" {N_LAT} \u00d7 {N_LON} = {N_LAT * N_LON:,} cells"),
               x=0.5, xanchor="center", font=dict(size=13)),
    legend=dict(yanchor="bottom", y=0.02, xanchor="left", x=0.02,
                bgcolor="rgba(255,255,255,0.85)"))

PLOT_DIR.mkdir(parents=True, exist_ok=True)
fig.write_html(str(PLOT_DIR / f"era5_area_{AIRPORT}_{TARGET_DATE}.html"))
fig.show()

## 6. Detail map: the ERA5 grid at true resolution

Zoomed to a `DETAIL_HALF_WIDTH_DEG` window around the airport so the actual
0.25° grid lines and cell corners are legible. Drawing them across the whole
request area would be several hundred lines and tens of thousands of corners.

In [13]:
ap_lat, ap_lon = AIRPORT_LATLON[AIRPORT]
d = DETAIL_HALF_WIDTH_DEG
d_n = min(NORTH, math.ceil((ap_lat + d) / GRID_STEP_DEG) * GRID_STEP_DEG)
d_s = max(SOUTH, math.floor((ap_lat - d) / GRID_STEP_DEG) * GRID_STEP_DEG)
d_w = max(WEST, math.floor((ap_lon - d) / GRID_STEP_DEG) * GRID_STEP_DEG)
d_e = min(EAST, math.ceil((ap_lon + d) / GRID_STEP_DEG) * GRID_STEP_DEG)

near = ((lats >= d_s) & (lats <= d_n) & (lons >= d_w) & (lons <= d_e))
cx, cy = grid_corners(d_n, d_w, d_s, d_e, GRID_STEP_DEG)
gl_lon, gl_lat = grid_lines(d_n, d_w, d_s, d_e, GRID_STEP_DEG, 1)

fig2 = go.Figure()

fig2.add_trace(go.Scattermap(
    lat=gl_lat, lon=gl_lon, mode="lines",
    line=dict(width=0.8, color="rgba(60,60,60,0.55)"),
    name=f"ERA5 grid ({GRID_STEP_DEG}\u00b0)", hoverinfo="skip"))

fig2.add_trace(go.Scattermap(
    lat=cy, lon=cx, mode="markers",
    marker=dict(size=5, color="#e15759"),
    name=f"Grid cell corners (n={len(cx):,})",
    hovertemplate="%{lat:.2f}, %{lon:.2f}<extra></extra>"))

fig2.add_trace(go.Scattermap(
    lat=lats[near], lon=lons[near],
    mode="markers", marker=dict(size=4, color="#2c7fb8", opacity=0.5),
    name=f"Trajectory points (n={int(near.sum()):,})", hoverinfo="skip"))

fig2.add_trace(go.Scattermap(
    lat=[ap_lat], lon=[ap_lon], mode="markers+text",
    marker=dict(size=14, color="black"),
    text=[AIRPORT], textposition="top right",
    textfont=dict(size=13), name=AIRPORT))

fig2.update_layout(
    map=dict(style=MAP_STYLE,
             center=dict(lat=ap_lat, lon=ap_lon), zoom=6.4),
    margin=dict(r=0, t=52, l=0, b=0), height=680,
    title=dict(text=(f"ERA5 {GRID_STEP_DEG}\u00b0 grid near {AIRPORT} "
                     f"&mdash; {TARGET_DATE}"),
               x=0.5, xanchor="center", font=dict(size=13)),
    legend=dict(yanchor="bottom", y=0.02, xanchor="left", x=0.02,
                bgcolor="rgba(255,255,255,0.85)"))

fig2.write_html(str(PLOT_DIR / f"era5_grid_detail_{AIRPORT}_{TARGET_DATE}.html"))
fig2.show()